In [ ]:
%pip install transformers accelerate

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()
print(clip.available_models())
print("CLIP loaded on:", device)

100%|███████████████████████████████████████| 338M/338M [00:12<00:00, 27.7MiB/s]


['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']
CLIP loaded on: cuda


In [4]:
import json
from pathlib import Path
from collections import Counter

DATA_3RSCAN_ROOT = Path("/home/station2/Zermat/RTSGS-SLAM/Datasets/3Rscan/scans")
DATA_3DSSG_ROOT = Path("/home/station2/Zermat/RTSGS-SLAM/Datasets/3DSSG")

with open(DATA_3DSSG_ROOT / "objects.json", "r") as f:
    data1 = json.load(f)

with open(DATA_3DSSG_ROOT / "relationships.json", "r") as f:
    rel_data = json.load(f)

objects_scans = {s["scan"]: s for s in data1.get("scans", [])}
rel_scans = {s["scan"]: s for s in rel_data.get("scans", [])}

# Collect scans that have both 3DSSG annotations and 3RScan semseg.v2.json
all_candidate_scan_ids = sorted(set(objects_scans.keys()) & set(rel_scans.keys()))
scan_ids = []
for sid in all_candidate_scan_ids:
    semseg_path = DATA_3RSCAN_ROOT / sid / "semseg.v2.json"
    if semseg_path.exists():
        scan_ids.append(sid)

if not scan_ids:
    raise ValueError("No valid scans found with objects + relationships + semseg.v2.json")

PREVIEW_SCAN_ID = scan_ids[0]
TARGET_SCAN_ID = PREVIEW_SCAN_ID  # kept for compatibility with visualization cells

print("Indexed multi-scan dataset:")
print(f"  3DSSG object scans      : {len(objects_scans)}")
print(f"  3DSSG relationship scans: {len(rel_scans)}")
print(f"  usable scans            : {len(scan_ids)}")
print(f"  preview scan            : {PREVIEW_SCAN_ID}")

Indexed multi-scan dataset:
  3DSSG object scans      : 1482
  3DSSG relationship scans: 1335
  usable scans            : 1335
  preview scan            : 02b33df9-be2b-2d54-9062-1253be3ce186


In [5]:
def build_clip_text(obj):
    label = obj.get("label", "object")
    affordances = obj.get("affordances", [])
    attributes = obj.get("attributes", {})

    def join_attr(key):
        vals = attributes.get(key, [])
        return ", ".join(vals) if isinstance(vals, list) and vals else ""

    color = join_attr("color")
    material = join_attr("material")
    shape = join_attr("shape")
    size = join_attr("size")
    texture = join_attr("texture")
    style = join_attr("style")
    state = join_attr("state")
    lexical = join_attr("lexical")
    other = join_attr("other")

    sentence = "This is "
    sentence += f"a {size} " if size else "a "

    if color:
        sentence += f"{color} "
    if material:
        sentence += f"{material} "

    sentence += f"{label}"

    attr_parts = []
    if shape:
        attr_parts.append(f"{shape} shape")
    if texture:
        attr_parts.append(f"{texture} texture")
    if style:
        attr_parts.append(f"{style} style")
    if state:
        attr_parts.append(f"in {state} state")

    if attr_parts:
        sentence += ", with " + ", ".join(attr_parts)

    extra = []
    if lexical:
        extra.append(lexical)
    if other:
        extra.append(other)

    if extra:
        sentence += ", described as " + ", ".join(extra)

    if affordances:
        sentence += ", and it can be used for " + ", ".join(affordances)

    sentence += "."
    return sentence

def to_int_if_possible(x):
    if isinstance(x, int):
        return x
    if isinstance(x, str) and x.isdigit():
        return int(x)
    return None

def build_scan_bundle(scan_id):
    semseg_path = DATA_3RSCAN_ROOT / scan_id / "semseg.v2.json"
    with open(semseg_path, "r") as f:
        semseg_data = json.load(f)

    scan_objects_data = objects_scans[scan_id]
    scan_rel_data = rel_scans.get(scan_id, {})

    objects = scan_objects_data.get("objects", [])
    seg_groups = semseg_data.get("segGroups", [])
    seg_lookup = {seg.get("objectId"): seg for seg in seg_groups if "objectId" in seg}

    merged_objects = []
    missing_ids = []

    for obj in objects:
        obj_id = to_int_if_possible(obj.get("id"))
        merged_obj = obj.copy()

        if obj_id in seg_lookup:
            seg = seg_lookup[obj_id].copy()
            seg.pop("segments", None)
            merged_obj["segmentation"] = seg
        else:
            merged_obj["segmentation"] = None
            missing_ids.append(obj_id)

        merged_obj["clip_text"] = build_clip_text(merged_obj)
        merged_objects.append(merged_obj)

    id_to_label = {}
    for obj in merged_objects:
        obj_id = to_int_if_possible(obj.get("id"))
        if obj_id is not None:
            id_to_label[obj_id] = obj.get("label")

    scan_relationships_raw = scan_rel_data.get("relationships", [])
    scan_relationships = []
    for rel in scan_relationships_raw:
        if not isinstance(rel, list) or len(rel) < 4:
            continue
        subject_id, object_id, predicate_id, predicate = rel[:4]
        scan_relationships.append({
            "subject_id": subject_id,
            "subject_label": id_to_label.get(subject_id),
            "object_id": object_id,
            "object_label": id_to_label.get(object_id),
            "predicate_id": predicate_id,
            "predicate": predicate,
        })

    return {
        "scan": scan_id,
        "objects": merged_objects,
        "relationships": scan_relationships,
        "missing_ids": missing_ids,
    }

# Build one preview scan bundle for visualization/search cells
preview_bundle = build_scan_bundle(PREVIEW_SCAN_ID)
merged_objects = preview_bundle["objects"]
scan_relationships = preview_bundle["relationships"]

print("Preview scan prepared:")
print(f"  scan: {PREVIEW_SCAN_ID}")
print(f"  objects: {len(merged_objects)}")
print(f"  relationships: {len(scan_relationships)}")
print(f"  missing segmentation IDs: {len(preview_bundle['missing_ids'])}")

Preview scan prepared:
  scan: 02b33df9-be2b-2d54-9062-1253be3ce186
  objects: 26
  relationships: 325
  missing segmentation IDs: 0


In [6]:
all_attr_keys = []
attr_key_per_object = []

# iterate over ALL scans
for scan in data1["scans"]:
    for obj in scan["objects"]:
        attrs = obj.get("attributes", {})
        keys = list(attrs.keys())

        all_attr_keys.extend(keys)
        attr_key_per_object.append(len(keys))

# unique attribute names
unique_keys = sorted(set(all_attr_keys))

print("\n===== UNIQUE ATTRIBUTE NAMES =====")
for k in unique_keys:
    print(k)

print(f"\nTotal unique attribute types: {len(unique_keys)}")

# =========================
# FREQUENCY (important)
# =========================
print("\n===== ATTRIBUTE FREQUENCY =====")
for k, v in Counter(all_attr_keys).most_common():
    print(f"{k:20s} {v}")

# =========================
# OBJECT ATTRIBUTE STATS
# =========================
print("\n===== ATTRIBUTES PER OBJECT =====")
print(f"Avg attributes per object : {sum(attr_key_per_object)/len(attr_key_per_object):.2f}")
print(f"Max attributes in object  : {max(attr_key_per_object)}")
print(f"Min attributes in object  : {min(attr_key_per_object)}")


===== UNIQUE ATTRIBUTE NAMES =====
color
lexical
material
other
shape
size
state
style
texture

Total unique attribute types: 9

===== ATTRIBUTE FREQUENCY =====
lexical              20337
color                14319
shape                13499
size                 10183
material             6211
state                4141
other                3516
texture              2757
style                93

===== ATTRIBUTES PER OBJECT =====
Avg attributes per object : 1.73
Max attributes in object  : 7
Min attributes in object  : 0


In [7]:
import numpy as np

def get_clip_embedding(text):
    with torch.no_grad():
        tokens = clip.tokenize([text]).to(device)
        emb = model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy()[0]

# Set to an integer for quick debug runs (e.g., 10), or None for all scans
MAX_SCANS_FOR_EXPORT = None
selected_scan_ids = scan_ids if MAX_SCANS_FOR_EXPORT is None else scan_ids[:MAX_SCANS_FOR_EXPORT]

all_scan_bundles = []
for i, sid in enumerate(selected_scan_ids, start=1):
    bundle = build_scan_bundle(sid)

    for obj in bundle["objects"]:
        obj["clip_embedding"] = get_clip_embedding(obj["clip_text"]).tolist()

    all_scan_bundles.append({
        "scan": sid,
        "objects": bundle["objects"],
        "relationships": bundle["relationships"],
    })

    if i == 1 or i % 20 == 0 or i == len(selected_scan_ids):
        print(f"Processed scans: {i}/{len(selected_scan_ids)}")

# Keep preview variables in sync for downstream visualization/search cells
preview_map = {s["scan"]: s for s in all_scan_bundles}
if PREVIEW_SCAN_ID in preview_map:
    merged_objects = preview_map[PREVIEW_SCAN_ID]["objects"]
    scan_relationships = preview_map[PREVIEW_SCAN_ID]["relationships"]

with open("data_all_scans.json", "w") as f:
    json.dump({"scans": all_scan_bundles}, f)

print("Saved data_all_scans.json ✅")
print(f"  total scans exported: {len(all_scan_bundles)}")

Processed scans: 1/1335
Processed scans: 20/1335
Processed scans: 40/1335
Processed scans: 60/1335
Processed scans: 80/1335
Processed scans: 100/1335
Processed scans: 120/1335
Processed scans: 140/1335
Processed scans: 160/1335
Processed scans: 180/1335
Processed scans: 200/1335
Processed scans: 220/1335
Processed scans: 240/1335
Processed scans: 260/1335
Processed scans: 280/1335
Processed scans: 300/1335
Processed scans: 320/1335
Processed scans: 340/1335
Processed scans: 360/1335
Processed scans: 380/1335
Processed scans: 400/1335
Processed scans: 420/1335
Processed scans: 440/1335
Processed scans: 460/1335
Processed scans: 480/1335
Processed scans: 500/1335
Processed scans: 520/1335
Processed scans: 540/1335
Processed scans: 560/1335
Processed scans: 580/1335
Processed scans: 600/1335
Processed scans: 620/1335
Processed scans: 640/1335
Processed scans: 660/1335
Processed scans: 680/1335
Processed scans: 700/1335
Processed scans: 720/1335
Processed scans: 740/1335
Processed scans: 7

In [8]:
import json

KEEP_KEYS = {
    "id",
    "global_id",
    "label",
    "clip_text",
    "clip_embedding",
}

REL_KEEP_KEYS = {
    "subject_id",
    "subject_label",
    "object_id",
    "object_label",
    "predicate_id",
    "predicate",
}

def simplify_object(obj):
    out = {}
    for k in KEEP_KEYS:
        if k in obj:
            out[k] = obj[k]

    seg = obj.get("segmentation", {})
    if isinstance(seg, dict):
        if "obb" in seg:
            out["obb"] = seg["obb"]
        if "dominantNormal" in seg:
            out["dominantNormal"] = seg["dominantNormal"]

    return out

def simplify_relationship(rel):
    out = {}
    for k in REL_KEEP_KEYS:
        if k in rel:
            out[k] = rel[k]
    return out

def simplify_scan(scan_bundle):
    return {
        "scan": scan_bundle.get("scan"),
        "objects": [simplify_object(o) for o in scan_bundle.get("objects", [])],
        "relationships": [simplify_relationship(r) for r in scan_bundle.get("relationships", [])],
    }

def print_json_tree(node, name="root", indent="", max_depth=4, depth=0):
    if isinstance(node, dict):
        print(f"{indent}{name}: {{object}}")
        if depth >= max_depth:
            print(f"{indent}  ...")
            return
        for key, value in node.items():
            print_json_tree(value, name=key, indent=indent + "  ", max_depth=max_depth, depth=depth + 1)
    elif isinstance(node, list):
        print(f"{indent}{name}: [array] (len={len(node)})")
        if depth >= max_depth:
            print(f"{indent}  ...")
            return
        if node:
            print_json_tree(node[0], name="item", indent=indent + "  ", max_depth=max_depth, depth=depth + 1)
    else:
        print(f"{indent}{name}: <{type(node).__name__}>")

with open("data_all_scans.json", "r") as f:
    data = json.load(f)

simplified_scans = [simplify_scan(s) for s in data.get("scans", [])]
simplified = {"scans": simplified_scans}

with open("clean_data_all_scans.json", "w") as f:
    json.dump(simplified, f, indent=4, ensure_ascii=False)

total_objects = sum(len(s.get("objects", [])) for s in simplified_scans)
total_relationships = sum(len(s.get("relationships", [])) for s in simplified_scans)

print("Saved clean_data_all_scans.json ✅")
print(f"  scans: {len(simplified_scans)}")
print(f"  objects: {total_objects}")
print(f"  relationships: {total_relationships}")

if simplified_scans:
    print("\nJSON tree (schema preview, first scan, no values):")
    print_json_tree(simplified_scans[0], name="scan_item", max_depth=4)

Saved clean_data_all_scans.json ✅
  scans: 1335
  objects: 38929
  relationships: 543956

JSON tree (schema preview, first scan, no values):
scan_item: {object}
  scan: <str>
  objects: [array] (len=26)
    item: {object}
      clip_text: <str>
      clip_embedding: [array] (len=512)
        item: <float>
      label: <str>
      global_id: <str>
      id: <str>
      obb: {object}
        axesLengths: [array] (len=3)
          ...
        centroid: [array] (len=3)
          ...
        normalizedAxes: [array] (len=9)
          ...
      dominantNormal: [array] (len=3)
        item: <int>
  relationships: [array] (len=325)
    item: {object}
      object_id: <int>
      object_label: <str>
      subject_id: <int>
      predicate: <str>
      predicate_id: <int>
      subject_label: <str>


In [9]:
import numpy as np
import plotly.graph_objects as go

def _extract_topdown_position(obj):
    """Return (x, z) center for top-down view from segmentation OBB data."""
    seg = obj.get("segmentation") or {}
    obb = seg.get("obb") if isinstance(seg, dict) else None

    if isinstance(obb, dict):
        for key in ("centroid", "center", "position"):
            if key in obb and isinstance(obb[key], (list, tuple)) and len(obb[key]) >= 3:
                c = obb[key]
                return float(c[0]), float(c[2])

    return None

def plot_topdown_relationships(max_relationships=120, annotate_predicates=False):
    if "merged_objects" not in globals():
        raise RuntimeError("merged_objects is not available. Run the merge/build cells first.")
    if "scan_relationships" not in globals():
        raise RuntimeError("scan_relationships is not available. Run the embedding/export cell first.")

    id_to_obj = {}
    positions = {}

    for obj in merged_objects:
        obj_id = obj.get("id")
        if not str(obj_id).isdigit():
            continue
        obj_id = int(obj_id)
        id_to_obj[obj_id] = obj
        pos = _extract_topdown_position(obj)
        if pos is not None:
            positions[obj_id] = pos

    if not positions:
        raise RuntimeError("No object positions found in OBB data for top-down plotting.")

    fig = go.Figure()

    # Object points + hover
    obj_ids = list(positions.keys())
    obj_x = [positions[i][0] for i in obj_ids]
    obj_y = [positions[i][1] for i in obj_ids]
    obj_labels = [id_to_obj.get(i, {}).get("label", "obj") for i in obj_ids]

    fig.add_trace(
        go.Scatter(
            x=obj_x,
            y=obj_y,
            mode="markers+text",
            text=[f"{lbl} ({oid})" for lbl, oid in zip(obj_labels, obj_ids)],
            textposition="top center",
            marker=dict(size=8, color="#1f77b4", line=dict(width=0.6, color="black")),
            name="Objects",
            customdata=np.array(list(zip(obj_ids, obj_labels)), dtype=object),
            hovertemplate="Object ID: %{customdata[0]}<br>Label: %{customdata[1]}<extra></extra>",
        )
    )

    # Directed relationships: arrow + hover marker at edge midpoint
    drawn = 0
    for rel in scan_relationships:
        if drawn >= max_relationships:
            break

        sid = rel.get("subject_id")
        oid = rel.get("object_id")
        pred = rel.get("predicate", "rel")

        if sid not in positions or oid not in positions:
            continue

        x1, y1 = positions[sid]
        x2, y2 = positions[oid]

        # Relationship line
        fig.add_trace(
            go.Scatter(
                x=[x1, x2],
                y=[y1, y2],
                mode="lines",
                line=dict(color="rgba(255,127,14,0.35)", width=1.6),
                hoverinfo="skip",
                showlegend=False,
            )
        )

        # Arrow head to show direction (subject -> object)
        fig.add_annotation(
            x=x2,
            y=y2,
            ax=x1,
            ay=y1,
            xref="x",
            yref="y",
            axref="x",
            ayref="y",
            showarrow=True,
            arrowhead=3,
            arrowsize=1,
            arrowwidth=1.2,
            arrowcolor="rgba(255,127,14,0.45)",
            opacity=0.9
        )

        # Invisible midpoint marker for hover relationship type
        mx, my = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        fig.add_trace(
            go.Scatter(
                x=[mx],
                y=[my],
                mode="markers",
                marker=dict(size=10, color="rgba(0,0,0,0)"),
                customdata=[[sid, rel.get("subject_label"), oid, rel.get("object_label"), pred]],
                hovertemplate=(
                    "Subject: %{customdata[1]} (%{customdata[0]})<br>"
                    "Predicate: %{customdata[4]}<br>"
                    "Object: %{customdata[3]} (%{customdata[2]})<extra></extra>"
                ),
                showlegend=False,
            )
        )

        if annotate_predicates:
            fig.add_annotation(
                x=mx,
                y=my,
                text=pred,
                showarrow=False,
                font=dict(size=9, color="#d62728"),
                bgcolor="rgba(255,255,255,0.65)",
            )

        drawn += 1

    fig.update_layout(
        title=f"Top-Down Relationship View | Scan {TARGET_SCAN_ID} | Drawn relations: {drawn}",
        xaxis_title="X",
        yaxis_title="Z",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        template="plotly_white",
        height=850,
        hovermode="closest"
    )

    fig.show()

plot_topdown_relationships(max_relationships=120, annotate_predicates=False)

In [11]:
# ===============================
# Multi-Scan Training - Cell 1/3
# Data loading and graph tensors
# ===============================
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_3DSSG_ROOT = Path("/home/station2/Zermat/RTSGS-SLAM/Datasets/3DSSG")
CLEAN_MULTI_PATH = Path("clean_data_all_scans.json")

with open(CLEAN_MULTI_PATH, "r") as f:
    clean_multi = json.load(f)

def read_nonempty_lines(path):
    with open(path, "r") as f:
        return [ln.strip() for ln in f if ln.strip()]

relationship_names = read_nonempty_lines(DATA_3DSSG_ROOT / "relationships.txt")
rel_to_idx = {name: i for i, name in enumerate(relationship_names)}
idx_to_rel = {i: name for name, i in rel_to_idx.items()}
num_rel_classes = len(relationship_names)

print("Loaded multi-scan clean data:")
print(f"  scans: {len(clean_multi.get('scans', []))}")
print(f"  relationship classes: {num_rel_classes}")

def to_int_if_possible(x):
    if isinstance(x, int):
        return x
    if isinstance(x, str) and x.isdigit():
        return int(x)
    return None

def extract_geom_vec(obj):
    obb = obj.get("obb", {}) if isinstance(obj.get("obb", {}), dict) else {}
    centroid = obb.get("centroid", [0.0, 0.0, 0.0])
    axes_lengths = obb.get("axesLengths", [0.0, 0.0, 0.0])
    normalized_axes = obb.get("normalizedAxes", [0.0] * 9)
    dominant_normal = obj.get("dominantNormal", [0.0, 0.0, 0.0])

    c = [float(v) for v in (centroid[:3] + [0.0] * 3)[:3]]
    a = [float(v) for v in (axes_lengths[:3] + [0.0] * 3)[:3]]
    n = [float(v) for v in (normalized_axes[:9] + [0.0] * 9)[:9]]
    d = [float(v) for v in (dominant_normal[:3] + [0.0] * 3)[:3]]
    return c + a + n + d

def standardize_features(x, eps=1e-6):
    mu = x.mean(dim=0, keepdim=True)
    sigma = x.std(dim=0, keepdim=True).clamp_min(eps)
    return (x - mu) / sigma

def build_knn_graph(centroids, k=6):
    n = centroids.shape[0]
    if n <= 1:
        return torch.zeros((0, 2), dtype=torch.long)

    k_eff = min(k, n - 1)
    d = torch.cdist(centroids, centroids)
    d.fill_diagonal_(float("inf"))
    nn_idx = torch.topk(d, k_eff, largest=False, dim=1).indices

    src = torch.arange(n).unsqueeze(1).repeat(1, k_eff).reshape(-1)
    dst = nn_idx.reshape(-1)
    e = torch.stack([src, dst], dim=1)
    e_rev = e[:, [1, 0]]
    e_all = torch.cat([e, e_rev], dim=0)
    return torch.unique(e_all, dim=0)

def build_scan_graph(scan_bundle, neg_ratio=0.5):
    objects = scan_bundle.get("objects", [])
    relationships = scan_bundle.get("relationships", [])

    id_to_node = {}
    node_clip = []
    node_geom_raw = []

    for obj in objects:
        oid = to_int_if_possible(obj.get("id"))
        emb = obj.get("clip_embedding")
        if oid is None or not isinstance(emb, list) or len(emb) == 0:
            continue
        id_to_node[oid] = len(node_clip)
        node_clip.append([float(v) for v in emb])
        node_geom_raw.append(extract_geom_vec(obj))

    if len(node_clip) < 2:
        return None

    node_clip = torch.tensor(node_clip, dtype=torch.float32)
    node_geom_raw = torch.tensor(node_geom_raw, dtype=torch.float32)

    # Safety + normalization to prevent optimization blow-ups
    node_clip = torch.nan_to_num(node_clip, nan=0.0, posinf=0.0, neginf=0.0)
    node_geom_raw = torch.nan_to_num(node_geom_raw, nan=0.0, posinf=0.0, neginf=0.0)
    node_geom = standardize_features(node_geom_raw)
    node_clip = standardize_features(node_clip)

    n_nodes = node_clip.shape[0]

    edge_to_targets = {}
    unknown_predicates = 0

    for rel in relationships:
        sid = to_int_if_possible(rel.get("subject_id"))
        oid = to_int_if_possible(rel.get("object_id"))
        pred = rel.get("predicate")

        if sid not in id_to_node or oid not in id_to_node or sid == oid:
            continue
        if pred not in rel_to_idx:
            unknown_predicates += 1
            continue

        pair = (id_to_node[sid], id_to_node[oid])
        if pair not in edge_to_targets:
            edge_to_targets[pair] = torch.zeros(num_rel_classes, dtype=torch.float32)
        edge_to_targets[pair][rel_to_idx[pred]] = 1.0

    all_pairs = [(i, j) for i in range(n_nodes) for j in range(n_nodes) if i != j]
    pos_pairs = list(edge_to_targets.keys())
    neg_pairs = [p for p in all_pairs if p not in edge_to_targets]

    num_neg = min(len(neg_pairs), int(len(pos_pairs) * neg_ratio))
    sampled_neg = random.sample(neg_pairs, num_neg) if num_neg > 0 else []

    query_pairs = pos_pairs + sampled_neg
    if not query_pairs:
        return None

    query_pairs_t = torch.tensor(query_pairs, dtype=torch.long)
    targets = []
    for p in query_pairs:
        if p in edge_to_targets:
            targets.append(edge_to_targets[p])
        else:
            targets.append(torch.zeros(num_rel_classes, dtype=torch.float32))
    targets_t = torch.stack(targets, dim=0)

    centroids = node_geom_raw[:, :3]
    graph_edges = build_knn_graph(centroids, k=6)

    return {
        "scan": scan_bundle.get("scan"),
        "node_clip": node_clip,
        "node_geom": node_geom,
        "node_geom_raw": node_geom_raw,
        "graph_edges": graph_edges,
        "query_pairs": query_pairs_t,
        "targets": targets_t,
        "unknown_predicates": unknown_predicates,
    }

scan_graphs = []
for scan_bundle in clean_multi.get("scans", []):
    g = build_scan_graph(scan_bundle, neg_ratio=0.5)
    if g is not None:
        scan_graphs.append(g)

num_nodes_total = sum(g["node_clip"].shape[0] for g in scan_graphs)
num_edges_total = sum(g["query_pairs"].shape[0] for g in scan_graphs)
total_pos_labels = sum(int(g["targets"].sum().item()) for g in scan_graphs)
print("Prepared training graphs:")
print(f"  usable scans: {len(scan_graphs)}")
print(f"  total nodes: {num_nodes_total}")
print(f"  total query edges: {num_edges_total}")
print(f"  positive relation labels: {total_pos_labels}")

Loaded multi-scan clean data:
  scans: 1335
  relationship classes: 41
Prepared training graphs:
  usable scans: 1335
  total nodes: 38929
  total query edges: 480493
  positive relation labels: 543953


In [12]:
# ===============================
# Multi-Scan Training - Cell 2/3
# Model definition
# ===============================
class MPNNLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.msg_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.upd_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, h, edges):
        if edges.numel() == 0:
            return h
        src, dst = edges[:, 0], edges[:, 1]
        msg = self.msg_mlp(torch.cat([h[src], h[dst]], dim=-1))

        agg = torch.zeros_like(h)
        agg.index_add_(0, dst, msg)

        h_new = self.upd_mlp(torch.cat([h, agg], dim=-1))
        return h + h_new

class RelationMPGNN(nn.Module):
    def __init__(self, clip_dim, geom_dim, hidden_dim, rel_classes, num_layers=3, dropout=0.1):
        super().__init__()
        self.clip_proj = nn.Sequential(
            nn.Linear(clip_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
        )

        self.node_fuse = nn.Sequential(
            nn.Linear(256 + geom_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.layers = nn.ModuleList([MPNNLayer(hidden_dim) for _ in range(num_layers)])

        # Rich relative edge features: delta xyz, log distance,
        # log size ratios, dominant-normal cosine similarity.
        self.rel_feat_dim = 8
        self.edge_head = nn.Sequential(
            nn.Linear(2 * hidden_dim + self.rel_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rel_classes),
        )

    def encode_nodes(self, clip_x, geom_x, graph_edges):
        clip_z = self.clip_proj(clip_x)
        h = self.node_fuse(torch.cat([clip_z, geom_x], dim=-1))
        for layer in self.layers:
            h = layer(h, graph_edges)
        return h

    def build_edge_features(self, geom_raw_x, query_pairs):
        s, o = query_pairs[:, 0], query_pairs[:, 1]

        c_s = geom_raw_x[s, :3]
        c_o = geom_raw_x[o, :3]
        delta = c_o - c_s

        dist = torch.norm(delta, dim=-1, keepdim=True)
        log_dist = torch.log1p(dist)

        size_s = geom_raw_x[s, 3:6].clamp_min(1e-6)
        size_o = geom_raw_x[o, 3:6].clamp_min(1e-6)
        log_size_ratio = torch.log(size_o / size_s)

        n_s = geom_raw_x[s, 15:18]
        n_o = geom_raw_x[o, 15:18]
        normal_cos = F.cosine_similarity(n_s, n_o, dim=-1, eps=1e-8).unsqueeze(-1)

        return torch.cat([delta, log_dist, log_size_ratio, normal_cos], dim=-1)

    def forward(self, clip_x, geom_x, graph_edges, query_pairs, geom_raw_x=None):
        h = self.encode_nodes(clip_x, geom_x, graph_edges)
        s, o = query_pairs[:, 0], query_pairs[:, 1]
        geom_for_edges = geom_x if geom_raw_x is None else geom_raw_x
        rel_feat = self.build_edge_features(geom_for_edges, query_pairs)
        edge_feat = torch.cat([h[s], h[o], rel_feat], dim=-1)
        logits = self.edge_head(edge_feat)
        return logits

assert len(scan_graphs) > 0, "No scan graphs prepared. Run the previous cell first."
clip_dim = scan_graphs[0]["node_clip"].shape[1]
geom_dim = scan_graphs[0]["node_geom"].shape[1]

model = RelationMPGNN(
    clip_dim=clip_dim,
    geom_dim=geom_dim,
    hidden_dim=256,
    rel_classes=num_rel_classes,
    num_layers=3,
    dropout=0.1,
).to(DEVICE)

print("Model ready:")
print(f"  clip_dim={clip_dim}, geom_dim={geom_dim}, rel_classes={num_rel_classes}")
print("  edge features: delta xyz + log distance + log size ratios + normal cosine")

Model ready:
  clip_dim=512, geom_dim=18, rel_classes=41
  edge features: delta xyz + log distance + log size ratios + normal cosine


In [13]:
# ===============================
# Multi-Scan Training - Cell 3/3
# Train loop (manual run)
# ===============================
from pathlib import Path
from tqdm.auto import tqdm


def split_scans_train_val(graphs, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    idx = list(range(len(graphs)))
    rng.shuffle(idx)
    n_val = max(1, int(len(idx) * val_ratio)) if len(idx) > 1 else 0
    val_idx = set(idx[:n_val])
    train_graphs = [g for i, g in enumerate(graphs) if i not in val_idx]
    val_graphs = [g for i, g in enumerate(graphs) if i in val_idx]
    return train_graphs, val_graphs


def compute_pos_weight(train_graphs, min_count=1.0, max_weight=50.0):
    pos_counts = torch.zeros(num_rel_classes, dtype=torch.float32)
    total_edges = 0.0
    for g in train_graphs:
        y = g["targets"]
        pos_counts += y.sum(dim=0)
        total_edges += y.shape[0]

    neg_counts = total_edges - pos_counts
    pos_weight = (neg_counts + 1.0) / (pos_counts + min_count)
    pos_weight = torch.clamp(pos_weight, max=max_weight)
    return pos_weight


def _f1_scores(pred, y, eps=1e-8):
    # Micro F1 over all labels
    tp = (pred * y).sum()
    fp = (pred * (1.0 - y)).sum()
    fn = ((1.0 - pred) * y).sum()
    micro_p = tp / (tp + fp + eps)
    micro_r = tp / (tp + fn + eps)
    micro_f1 = (2.0 * micro_p * micro_r) / (micro_p + micro_r + eps)

    # Macro F1 over relation classes
    tp_c = (pred * y).sum(dim=0)
    fp_c = (pred * (1.0 - y)).sum(dim=0)
    fn_c = ((1.0 - pred) * y).sum(dim=0)
    p_c = tp_c / (tp_c + fp_c + eps)
    r_c = tp_c / (tp_c + fn_c + eps)
    f1_c = (2.0 * p_c * r_c) / (p_c + r_c + eps)

    # Exclude classes with no support in y from macro average
    support = y.sum(dim=0)
    valid = support > 0
    macro_f1 = f1_c[valid].mean() if valid.any() else torch.tensor(0.0, device=y.device)

    return float(micro_f1.item()), float(macro_f1.item())


def evaluate_multilabel(model, graphs, criterion=None, threshold=0.5):
    model.eval()
    total_loss = 0.0
    total = 0
    total_exact = 0
    total_pos_recall_num = 0.0
    total_pos_recall_den = 0.0

    all_pred = []
    all_y = []

    with torch.no_grad():
        for g in graphs:
            clip_x = g["node_clip"].to(DEVICE)
            geom_x = g["node_geom"].to(DEVICE)
            geom_raw_x = g["node_geom_raw"].to(DEVICE)
            graph_e = g["graph_edges"].to(DEVICE)
            pairs = g["query_pairs"].to(DEVICE)
            y = g["targets"].to(DEVICE)

            logits = model(clip_x, geom_x, graph_e, pairs, geom_raw_x=geom_raw_x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=20.0, neginf=-20.0)

            if criterion is None:
                loss = F.binary_cross_entropy_with_logits(logits, y)
            else:
                loss = criterion(logits, y)
            total_loss += float(loss.item())

            pred = (torch.sigmoid(logits) >= threshold).float()
            exact = (pred == y).all(dim=1).float().sum().item()
            total_exact += exact
            total += y.shape[0]

            pos_mask = y.sum(dim=1) > 0
            if pos_mask.any():
                y_pos = y[pos_mask]
                p_pos = pred[pos_mask]
                tp = (p_pos * y_pos).sum().item()
                denom = y_pos.sum().item()
                total_pos_recall_num += tp
                total_pos_recall_den += denom

            all_pred.append(pred)
            all_y.append(y)

    mean_loss = total_loss / max(1, len(graphs))
    exact_acc = total_exact / max(1, total)
    pos_recall = total_pos_recall_num / max(1e-8, total_pos_recall_den)

    if all_pred:
        pred_cat = torch.cat(all_pred, dim=0)
        y_cat = torch.cat(all_y, dim=0)
        micro_f1, macro_f1 = _f1_scores(pred_cat, y_cat)
    else:
        micro_f1, macro_f1 = 0.0, 0.0

    return {
        "loss": mean_loss,
        "exact_acc": exact_acc,
        "pos_recall": pos_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def sweep_thresholds(model, graphs, thresholds=None):
    if thresholds is None:
        thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

    best = None
    for t in thresholds:
        m = evaluate_multilabel(model, graphs, criterion=None, threshold=t)
        row = {
            "threshold": float(t),
            "exact_acc": m["exact_acc"],
            "micro_f1": m["micro_f1"],
            "macro_f1": m["macro_f1"],
            "pos_recall": m["pos_recall"],
        }
        if best is None or row["macro_f1"] > best["macro_f1"]:
            best = row

    return best


def _save_checkpoint(path, model, optimizer, epoch, train_metrics, val_metrics=None, best_score=None):
    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "best_score": best_score,
    }
    torch.save(ckpt, path)


def _metric_for_selection(metrics, metric_name):
    if metric_name in metrics:
        return metrics[metric_name]
    # Backward-compatibility aliases
    if metric_name == "val_exact":
        return metrics.get("exact_acc", float("-inf"))
    if metric_name == "val_pos_recall":
        return metrics.get("pos_recall", float("-inf"))
    if metric_name == "val_micro_f1":
        return metrics.get("micro_f1", float("-inf"))
    if metric_name == "val_macro_f1":
        return metrics.get("macro_f1", float("-inf"))
    return float("-inf")


def train_multiscan(
    model,
    graphs,
    epochs=60,
    lr=3e-4,
    weight_decay=1e-4,
    val_ratio=0.15,
    checkpoint_dir="checkpoints/scenegraph_multiscan",
    checkpoint_every=5,
    threshold=0.5,
    save_best_on="val_macro_f1",
    resume_from="best.pt",
    resume_optimizer=False,
    lr_factor=0.8,
    lr_patience=10,
    min_lr=5e-5,
):
    train_graphs, val_graphs = split_scans_train_val(graphs, val_ratio=val_ratio, seed=SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    pos_weight = compute_pos_weight(train_graphs).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    ckpt_dir = Path(checkpoint_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=lr_factor,
        patience=lr_patience,
        min_lr=min_lr,
    )

    start_epoch = 1
    best_score = float("-inf")

    # Resume from checkpoint if requested and file exists
    resume_path = None
    if resume_from:
        candidate = Path(resume_from)
        resume_path = candidate if candidate.is_absolute() else (ckpt_dir / candidate)
        if resume_path.exists():
            ckpt = torch.load(resume_path, map_location=DEVICE)
            model.load_state_dict(ckpt["model_state_dict"], strict=False)
            if resume_optimizer and "optimizer_state_dict" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            start_epoch = int(ckpt.get("epoch", 0)) + 1
            best_score = float(ckpt.get("best_score", float("-inf")))
            print(f"Resumed model from: {resume_path}")
            print(f"Starting at epoch: {start_epoch}")

    print("Training split:")
    print(f"  train scans: {len(train_graphs)}")
    print(f"  val scans  : {len(val_graphs)}")
    print(f"  checkpoints: {ckpt_dir}")
    print(f"  lr={lr}, threshold={threshold}")
    print(f"  pos_weight range: [{pos_weight.min().item():.3f}, {pos_weight.max().item():.3f}]")

    epoch_bar = tqdm(range(start_epoch, epochs + 1), desc="Training epochs", unit="epoch")

    for epoch in epoch_bar:
        model.train()
        random.shuffle(train_graphs)
        running_loss = 0.0
        valid_steps = 0

        scan_bar = tqdm(train_graphs, desc=f"Epoch {epoch:03d} scans", unit="scan", leave=False)
        for g in scan_bar:
            clip_x = g["node_clip"].to(DEVICE)
            geom_x = g["node_geom"].to(DEVICE)
            geom_raw_x = g["node_geom_raw"].to(DEVICE)
            graph_e = g["graph_edges"].to(DEVICE)
            pairs = g["query_pairs"].to(DEVICE)
            y = g["targets"].to(DEVICE)

            logits = model(clip_x, geom_x, graph_e, pairs, geom_raw_x=geom_raw_x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=20.0, neginf=-20.0)
            loss = criterion(logits, y)

            if not torch.isfinite(loss):
                scan_bar.set_postfix(skip="non-finite loss")
                continue

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            running_loss += loss.item()
            valid_steps += 1
            scan_bar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / max(1, valid_steps)
        tr_eval = evaluate_multilabel(model, train_graphs, criterion=criterion, threshold=threshold)

        if val_graphs:
            va_eval = evaluate_multilabel(model, val_graphs, criterion=criterion, threshold=threshold)
            score_key = save_best_on.replace("val_", "") if save_best_on.startswith("val_") else save_best_on
            current_score = _metric_for_selection(va_eval, score_key)
            scheduler.step(va_eval["loss"])
        else:
            va_eval = {"loss": float("nan"), "exact_acc": float("nan"), "pos_recall": float("nan"), "micro_f1": float("nan"), "macro_f1": float("nan")}
            score_key = save_best_on.replace("train_", "") if save_best_on.startswith("train_") else save_best_on
            current_score = _metric_for_selection(tr_eval, score_key)
            scheduler.step(tr_eval["loss"])

        current_lr = optimizer.param_groups[0]["lr"]
        epoch_bar.set_postfix(
            train_loss=f"{tr_eval['loss']:.4f}",
            test_loss=f"{va_eval['loss']:.4f}" if val_graphs else "n/a",
            train_acc=f"{tr_eval['exact_acc']:.3f}",
            test_acc=f"{va_eval['exact_acc']:.3f}" if val_graphs else "n/a",
            val_f1=f"{va_eval['macro_f1']:.3f}" if val_graphs else "n/a",
            lr=f"{current_lr:.1e}",
        )

        # Print richer metrics after every epoch
        print(
            f"Epoch {epoch:03d} | train_loss={tr_eval['loss']:.4f} | test_loss={va_eval['loss']:.4f} "
            f"| train_acc={tr_eval['exact_acc']:.3f} | test_acc={va_eval['exact_acc']:.3f} "
            f"| train_microF1={tr_eval['micro_f1']:.3f} | test_microF1={va_eval['micro_f1']:.3f} "
            f"| train_macroF1={tr_eval['macro_f1']:.3f} | test_macroF1={va_eval['macro_f1']:.3f} "
            f"| lr={current_lr:.1e}"
        )

        train_metrics = {
            "loss": train_loss,
            "exact_acc": tr_eval["exact_acc"],
            "pos_recall": tr_eval["pos_recall"],
            "micro_f1": tr_eval["micro_f1"],
            "macro_f1": tr_eval["macro_f1"],
        }
        val_metrics = {
            "loss": va_eval["loss"],
            "exact_acc": va_eval["exact_acc"],
            "pos_recall": va_eval["pos_recall"],
            "micro_f1": va_eval["micro_f1"],
            "macro_f1": va_eval["macro_f1"],
        } if val_graphs else None

        if checkpoint_every > 0 and epoch % checkpoint_every == 0:
            periodic_path = ckpt_dir / f"epoch_{epoch:03d}.pt"
            _save_checkpoint(periodic_path, model, optimizer, epoch, train_metrics, val_metrics, best_score)

        if current_score > best_score + 1e-6:
            best_score = current_score
            best_path = ckpt_dir / "best.pt"
            _save_checkpoint(best_path, model, optimizer, epoch, train_metrics, val_metrics, best_score)

    last_path = ckpt_dir / "last.pt"
    _save_checkpoint(last_path, model, optimizer, epoch, train_metrics, val_metrics, best_score)

    print(f"Saved final checkpoint: {last_path}")
    print(f"Saved best checkpoint : {ckpt_dir / 'best.pt'}")

    if val_graphs:
        best_thr = sweep_thresholds(model, val_graphs)
        print("Best threshold on validation (by macro F1):")
        print(
            f"  thr={best_thr['threshold']:.2f} | macroF1={best_thr['macro_f1']:.3f} "
            f"| microF1={best_thr['micro_f1']:.3f} | exact_acc={best_thr['exact_acc']:.3f}"
        )

    return model


print("Training cell ready. It is NOT running automatically.")
print("Run manually:")
print("  model = train_multiscan(model, scan_graphs, epochs=120, lr=3e-4, resume_from='best.pt', save_best_on='val_macro_f1')")

Training cell ready. It is NOT running automatically.
Run manually:
  model = train_multiscan(model, scan_graphs, epochs=120, lr=3e-4, resume_from='best.pt', save_best_on='val_macro_f1')


In [ ]:
model = train_multiscan(
    model,
    scan_graphs,
    epochs=250,
    lr=3e-4,
    resume_from="best.pt",
    save_best_on="val_macro_f1",
)

Training split:
  train scans: 1135
  val scans  : 200
  checkpoints: checkpoints/scenegraph_multiscan
  lr=0.0003, threshold=0.5
  pos_weight range: [3.443, 50.000]


Training epochs:   0%|          | 0/250 [00:00<?, ?epoch/s]

Epoch 001 scans:   0%|          | 0/1135 [00:00<?, ?scan/s]

Epoch 001 | train_loss=0.2717 | test_loss=0.2865 | train_acc=0.086 | test_acc=0.082 | train_microF1=0.365 | test_microF1=0.367 | train_macroF1=0.238 | test_macroF1=0.227 | lr=3.0e-04


Epoch 002 scans:   0%|          | 0/1135 [00:00<?, ?scan/s]

...

Epoch 144 scans:   0%|          | 0/1135 [00:00<?, ?scan/s]

Epoch 144 | train_loss=0.0439 | test_loss=0.1446 | train_acc=0.654 | test_acc=0.620 | train_microF1=0.763 | test_microF1=0.729 | train_macroF1=0.684 | test_macroF1=0.628 | lr=6.3e-05


Epoch 145 scans:   0%|          | 0/1135 [00:00<?, ?scan/s]

Epoch 145 | train_loss=0.0441 | test_loss=0.1412 | train_acc=0.647 | test_acc=0.615 | train_microF1=0.760 | test_microF1=0.726 | train_macroF1=0.674 | test_macroF1=0.616 | lr=6.3e-05


Epoch 146 scans:   0%|          | 0/1135 [00:00<?, ?scan/s]

In [11]:
# ============================================
# Benchmark-style metrics on validation split
# ============================================
# This approximates the commonly reported 3RScan relation metrics

def _safe_div(a, b, eps=1e-8):
    return float(a / (b + eps))

def evaluate_benchmark_style(model, graphs, threshold=0.5):
    model.eval()

    all_pred = []
    all_y = []

    with torch.no_grad():
        for g in graphs:
            clip_x = g["node_clip"].to(DEVICE)
            geom_x = g["node_geom"].to(DEVICE)
            geom_raw_x = g["node_geom_raw"].to(DEVICE)
            graph_e = g["graph_edges"].to(DEVICE)
            pairs = g["query_pairs"].to(DEVICE)
            y = g["targets"].to(DEVICE)

            logits = model(clip_x, geom_x, graph_e, pairs, geom_raw_x=geom_raw_x)
            pred = (torch.sigmoid(logits) >= threshold).float()

            all_pred.append(pred.cpu())
            all_y.append(y.cpu())

    if not all_pred:
        raise RuntimeError("No graphs available for benchmark evaluation.")

    pred = torch.cat(all_pred, dim=0)
    y = torch.cat(all_y, dim=0)

    # Trip: exact match over predicate vector per directed pair
    trip = (pred == y).all(dim=1).float().mean().item() * 100.0

    # Pred: label-wise accuracy over all predicate entries
    pred_acc = (pred == y).float().mean().item() * 100.0

    # Positive-edge subset (Trip* / Pred*)
    pos_edge_mask = y.sum(dim=1) > 0
    if pos_edge_mask.any():
        pred_pos = pred[pos_edge_mask]
        y_pos = y[pos_edge_mask]
        trip_star = (pred_pos == y_pos).all(dim=1).float().mean().item() * 100.0
        pred_star = (pred_pos == y_pos).float().mean().item() * 100.0
    else:
        trip_star = 0.0
        pred_star = 0.0

    # mR.Pred: mean recall over predicate classes with support
    tp_c = (pred * y).sum(dim=0)
    fn_c = ((1.0 - pred) * y).sum(dim=0)
    support_c = y.sum(dim=0)
    valid = support_c > 0
    recall_c = tp_c / (tp_c + fn_c + 1e-8)
    mR_pred = recall_c[valid].mean().item() * 100.0 if valid.any() else 0.0

    # Extra helpful metrics
    micro_f1, macro_f1 = _f1_scores(pred, y)
    pos_recall = _safe_div((pred * y).sum().item(), y.sum().item()) * 100.0

    return {
        "Trip": trip,
        "Pred": pred_acc,
        "Trip*": trip_star,
        "Pred*": pred_star,
        "mR.Pred": mR_pred,
        "MicroF1": micro_f1 * 100.0,
        "MacroF1": macro_f1 * 100.0,
        "PosRecall": pos_recall,
    }

# Evaluate on the same validation split used in training
_, val_graphs = split_scans_train_val(scan_graphs, val_ratio=0.15, seed=SEED)
best_thr_dict = globals().get("best_thr", None)
eval_threshold = float(best_thr_dict["threshold"]) if isinstance(best_thr_dict, dict) and "threshold" in best_thr_dict else 0.5
bench = evaluate_benchmark_style(model, val_graphs, threshold=eval_threshold)

print(f"Validation benchmark-style metrics (threshold={eval_threshold:.2f})")
for k, v in bench.items():
    print(f"  {k:10s}: {v:.2f}")

print("\nNote: Obj / Obj* / mR.Obj are not computed here because this notebook does not train an object classifier.")

Validation benchmark-style metrics (threshold=0.50)
  Trip      : 62.88
  Pred      : 98.22
  Trip*     : 53.05
  Pred*     : 97.67
  mR.Pred   : 89.83
  MicroF1   : 74.60
  MacroF1   : 62.73
  PosRecall : 93.86

Note: Obj / Obj* / mR.Obj are not computed here because this notebook does not train an object classifier.


In [43]:
# ============================================
# Predicted relationships on one test/val scan
# ============================================
# This cell uses the trained model and visualizes predicted relations
# in top-down view, similar to the earlier ground-truth visualization.

# Pick a deterministic validation/test scan index using the same split seed
_, _val_graphs = split_scans_train_val(scan_graphs, val_ratio=0.15, seed=SEED)
if not _val_graphs:
    raise RuntimeError("No validation/test scans available. Check split settings.")

test_graph = _val_graphs[0]
test_scan_id = test_graph["scan"]
print(f"Using test scan: {test_scan_id}")

# Rebuild full bundle to recover segmentation/positions for plotting
_test_bundle = build_scan_bundle(test_scan_id)
merged_objects = _test_bundle["objects"]
TARGET_SCAN_ID = test_scan_id

# Build node index -> object metadata from clean data for this scan
_scan_clean = next((s for s in clean_multi.get("scans", []) if s.get("scan") == test_scan_id), None)
if _scan_clean is None:
    raise RuntimeError(f"Scan {test_scan_id} not found in clean_multi.")

node_to_obj = {}
node_idx = 0
for obj in _scan_clean.get("objects", []):
    oid = to_int_if_possible(obj.get("id"))
    emb = obj.get("clip_embedding")
    if oid is None or not isinstance(emb, list) or len(emb) == 0:
        continue
    node_to_obj[node_idx] = {
        "id": oid,
        "label": obj.get("label", "obj"),
    }
    node_idx += 1

# Use best threshold from previous training sweep if available
pred_threshold = 0.5
if "best_thr" in globals() and isinstance(best_thr, dict) and "threshold" in best_thr:
    pred_threshold = float(best_thr["threshold"])

model.eval()
with torch.no_grad():
    clip_x = test_graph["node_clip"].to(DEVICE)
    geom_x = test_graph["node_geom"].to(DEVICE)
    geom_raw_x = test_graph["node_geom_raw"].to(DEVICE)
    graph_e = test_graph["graph_edges"].to(DEVICE)
    query_pairs = test_graph["query_pairs"].to(DEVICE)

    logits = model(clip_x, geom_x, graph_e, query_pairs, geom_raw_x=geom_raw_x)
    probs = torch.sigmoid(logits).cpu()

pred_relationships = []
for edge_i in range(query_pairs.shape[0]):
    s_idx = int(query_pairs[edge_i, 0].item())
    o_idx = int(query_pairs[edge_i, 1].item())

    if s_idx not in node_to_obj or o_idx not in node_to_obj:
        continue

    active_rel_idx = (probs[edge_i] >= pred_threshold).nonzero(as_tuple=False).flatten().tolist()
    for rel_i in active_rel_idx:
        pred_relationships.append({
            "subject_id": node_to_obj[s_idx]["id"],
            "subject_label": node_to_obj[s_idx]["label"],
            "object_id": node_to_obj[o_idx]["id"],
            "object_label": node_to_obj[o_idx]["label"],
            "predicate_id": int(rel_i),
            "predicate": idx_to_rel.get(int(rel_i), f"rel_{rel_i}"),
        })

scan_relationships = pred_relationships
print(f"Predicted relationships: {len(scan_relationships)} (threshold={pred_threshold:.2f})")

# Reuse your existing top-down interactive plotter
plot_topdown_relationships(max_relationships=200, annotate_predicates=False)

Using test scan: 02b33dfd-be2b-2d54-91d2-55454852009e
Predicted relationships: 457 (threshold=0.50)


Using test scan: 02b33dfd-be2b-2d54-91d2-55454852009e
Predicted relationships: 457 (threshold=0.50)
